# IS 445: Illinois professional licensing visualizations

This notebook uses **one instructor-approved dataset** to make **two separate Altair/Vega-Lite charts** and exports each chart as a self-contained HTML page for a GitHub Pages / Jekyll website.

- Plot 1: normalized stacked bars comparing recorded license statuses for the eight most frequent license categories.
- Plot 2: yearly counts of **original issue dates**, with a **dropdown** that changes which licensing category is shown.

**Important:** This is a randomized sample in the Fall 2022 course dataset, not a current census of all Illinois licenses. The statuses are historical values in the dataset, not today's live status. The notebook does not print or export individual names, addresses, or license numbers.

In [2]:
# In a fresh environment, run this cell once to install dependencies.
# Uncomment the line below if pandas or altair is missing.
# %pip install pandas altair

import pandas as pd
import altair as alt
from pathlib import Path

print('Pandas:', pd.__version__)
print('Altair:', alt.__version__)

Pandas: 2.1.4
Altair: 6.3.0


## 1. Load the data from a URL

The homework requires loading a dataset directly from a URL. We use the public course GitHub repository, selecting only the three columns needed for our charts.

In [3]:
DATA_URL = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/licenses_fall2022.csv'

# All data is loaded from a URL (not a file on your computer).
raw = pd.read_csv(DATA_URL, low_memory=False)
required = ['License Type', 'License Status', 'Original Issue Date']
missing = [column for column in required if column not in raw.columns]
if missing:
    raise ValueError(f'The dataset is missing columns: {missing}')

df = raw[required].copy()
print(f'Loaded {len(df):,} license records and {len(raw.columns)} source columns.')
df.head()

Loaded 10,000 license records and 31 source columns.


,License Type,License Status,Original Issue Date
0,DETECTIVE BOARD,NOT RENEWED,02/03/2020
1,DETECTIVE BOARD,NOT RENEWED,02/07/1995
2,COSMO,NOT RENEWED,02/28/1945
3,COSMO,ACTIVE,11/22/2011
4,COSMO,NOT RENEWED,07/12/1995


## 2. Clean and transform the data

We trim whitespace, remove missing categories and statuses, combine rare status values into **Other**, select the eight license types with the most records, and parse the original issuance dates. Only the resulting **aggregated counts** go into the charts. This reduces the number of chart rows and avoids publishing personal fields.

In [4]:
# Keep records with a known license type and recorded status.
df['License Type'] = df['License Type'].astype('string').str.strip()
df['License Status'] = df['License Status'].astype('string').str.strip()
df = df.dropna(subset=['License Type', 'License Status']).copy()
df = df[(df['License Type'] != '') & (df['License Status'] != '')].copy()

# Most-common licensing boards / types in the sample.
top_types = df['License Type'].value_counts().head(8).index.tolist()
df = df[df['License Type'].isin(top_types)].copy()

# Make the status legend readable with three groups.
df['Status Group'] = 'Other'
df.loc[df['License Status'].str.upper() == 'ACTIVE', 'Status Group'] = 'Active'
df.loc[df['License Status'].str.upper() == 'NOT RENEWED', 'Status Group'] = 'Not renewed'

# Parse dates for the time-series chart.
df['Issue Date'] = pd.to_datetime(df['Original Issue Date'], format='%m/%d/%Y', errors='coerce')
df['Issue Year'] = df['Issue Date'].dt.year

print('Selected types:', top_types)
print('Records after filtering:', len(df))

Selected types: ['DETECTIVE BOARD', 'COSMO', 'DENTAL', 'FUNERAL AND EMBALMER', 'DIETETIC AND NUTRITION', 'DESIGN FIRM', 'MASSAGE LICENSING BD', 'HOME INSPECTOR']
Records after filtering: 9727


## 3. Plot 1 — License status composition

The horizontal position encodes the share of records; the vertical position encodes the licensing category; color encodes whether its recorded status is active, not renewed, or some other status. A **100% stacked bar chart** makes comparisons possible even when categories have different sample sizes.

In [5]:
status_counts = (
    df.groupby(['License Type', 'Status Group'], observed=True)
      .size()
      .reset_index(name='Records')
)
status_counts['Share'] = (
    status_counts['Records']
    / status_counts.groupby('License Type')['Records'].transform('sum')
)

chart1 = (
    alt.Chart(status_counts)
      .mark_bar()
      .encode(
          x=alt.X('sum(Records):Q', stack='normalize',
                  axis=alt.Axis(format='.0%'), title='Share of records within license type'),
          y=alt.Y('License Type:N', sort=top_types, title='License type'),
          color=alt.Color('Status Group:N',
                          scale=alt.Scale(
                              domain=['Active', 'Not renewed', 'Other'],
                              range=['#267DA8', '#E6A23C', '#A1A1AA']),
                          title='Recorded status'),
          tooltip=[
              alt.Tooltip('License Type:N', title='License type'),
              alt.Tooltip('Status Group:N', title='Status'),
              alt.Tooltip('Records:Q', format=',', title='Records'),
              alt.Tooltip('Share:Q', format='.1%', title='Share')
          ]
      )
      .properties(
          title='Recorded license statuses by category (Fall 2022 sample)',
          width=700,
          height=350
      )
)
chart1

alt.Chart(...)

## 4. Plot 2 — Yearly original issue counts with a dropdown

The horizontal position encodes the issuance year; vertical position encodes the number of sample records originally issued in that year. A **dropdown selection parameter** changes which license type is shown. This is a genuine user-controlled filter, not merely Altair's built-in pan/zoom. Tooltips provide exact values.

In [6]:
trend = df[df['Issue Year'].between(1990, 2022, inclusive='both')].copy()
trend['Issue Year'] = trend['Issue Year'].astype(int)

by_type = (
    trend.groupby(['License Type', 'Issue Year'], observed=True)
         .size()
         .reset_index(name='Records')
)
overall = (
    trend.groupby('Issue Year')
         .size()
         .reset_index(name='Records')
)
overall['License Type'] = 'All top 8 types'
trend_counts = pd.concat([overall, by_type], ignore_index=True)

choice = alt.param(
    name='selected_license_type',
    value='All top 8 types',
    bind=alt.binding_select(
        options=['All top 8 types'] + top_types,
        name='Choose license type: '
    )
)

chart2 = (
    alt.Chart(trend_counts)
      .mark_line(point=True, strokeWidth=2.5, color='#267DA8')
      .encode(
          x=alt.X('Issue Year:Q',
                  scale=alt.Scale(domain=[1990, 2022]),
                  axis=alt.Axis(format='d', tickCount=9),
                  title='Original issue year'),
          y=alt.Y('Records:Q', title='Number of license records'),
          tooltip=[
              alt.Tooltip('License Type:N', title='License type'),
              alt.Tooltip('Issue Year:Q', format='d', title='Year'),
              alt.Tooltip('Records:Q', format=',', title='Records')
          ]
      )
      .add_params(choice)
      .transform_filter(alt.datum['License Type'] == choice)
      .properties(
          title='Original license issues by year (1990–2022)',
          width=700,
          height=350
      )
)
chart2

alt.Chart(...)

## 5. Export both visualizations

Run this cell **after running the cells above**. It writes two standalone HTML charts to the notebook's working directory. Put these files in the root of your GitHub Pages repository alongside `index.md` and `analysis.ipynb`.

In [7]:
chart1.save('chart1.html')
chart2.save('chart2.html')
# JSON specifications are optional but can be useful for checking the Vega-Lite output.
chart1.save('chart1.json')
chart2.save('chart2.json')

for file in ['chart1.html', 'chart2.html', 'chart1.json', 'chart2.json']:
    print(f'Created {Path(file).resolve()}')

Created /Users/richardlin/Downloads/IS445_Altair_Jekyll_Homework_Starter/chart1.html
Created /Users/richardlin/Downloads/IS445_Altair_Jekyll_Homework_Starter/chart2.html
Created /Users/richardlin/Downloads/IS445_Altair_Jekyll_Homework_Starter/chart1.json
Created /Users/richardlin/Downloads/IS445_Altair_Jekyll_Homework_Starter/chart2.json


## 6. Publishing checklist

1. Run **all notebook cells**, including the export cell.
2. Put `analysis.ipynb`, `index.md`, `chart1.html`, and `chart2.html` in your `YOUR_USERNAME.github.io` repository's root. The JSON files are optional.
3. Edit `index.md` and replace both instances of `YOUR_USERNAME` in the **The Analysis** link with your GitHub username.
4. In GitHub, choose **Settings → Pages → Deploy from a branch**, `main` and `/(root)`.
5. Open `https://YOUR_USERNAME.github.io/` and confirm the dropdown changes Plot 2.
6. Submit this website URL, not the GitHub repository URL.